In [2]:
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

In [ ]:
data = pd.read_excel('../../dataset/world prediction/PFAS concentration in soil.xlsx')
maize_env = pd.read_csv('../../dataset/world prediction/maize_soil_all.csv')
wheat_env = pd.read_csv('../../dataset/world prediction/wheat_soil_all.csv')

In [4]:
soil_lon = 'lon'
soil_lat = 'lat'

env_lon = 'lon'
env_lat = 'lat'

In [5]:
soil_coords = data[[soil_lon, soil_lat]].values

maize_coords = maize_env[[env_lon, env_lat]].values
wheat_coords = wheat_env[[env_lon, env_lat]].values

In [ ]:
maize_tree = cKDTree(maize_coords)
wheat_tree = cKDTree(wheat_coords)

In [ ]:
# =========================================================
# Search nearest neighbor
# threshold can adjusted according to the distance between soil and crop
# unit: degree
#
# 0.02 about ~2 km
# 0.05 about ~5 km
# 0.1  about ~10 km
# =========================================================

threshold = 0.05

# maize
maize_dist, _ = maize_tree.query(
    soil_coords,
    k=1
)

# wheat
wheat_dist, _ = wheat_tree.query(
    soil_coords,
    k=1
)

# =========================================================
# Determine crop type
# =========================================================

crop_type = []

for md, wd in zip(maize_dist, wheat_dist):

    maize_match = md <= threshold
    wheat_match = wd <= threshold

    if maize_match and wheat_match:
        crop_type.append('Both')

    elif maize_match:
        crop_type.append('Maize')

    elif wheat_match:
        crop_type.append('Wheat')

    else:
        crop_type.append('None')

# =========================================================
# save
# =========================================================

data['Crop_match'] = crop_type
data['Nearest_maize_dist'] = maize_dist
data['Nearest_wheat_dist'] = wheat_dist

# =========================================================
# 查看统计
# =========================================================

print(data['Crop_match'].value_counts())

Crop_match
Both     324
Maize    270
None      30
Name: count, dtype: int64


In [ ]:
data.to_excel(
    '../../dataset/world prediction/PFAS_with_crop_match.xlsx',
    index=False
)